In [1]:
import pandas as pd
import glob

# Define the path to your CSV files
file_path_pattern = '/root/github/CTI_Scripts/py_scripts/backtesting/historical_data/EURUSD/DAT_MT_EURUSD_M1_*.csv'

# Define the column names
column_names = ['<DATE>', '<TIME>', '<OPEN>', '<HIGH>', '<LOW>', '<CLOSE>', '<TICKVOL>', '<VOL>', '<SPREAD>']

# Load and combine all CSV files
all_files = glob.glob(file_path_pattern)
data_list = []

for file in all_files:
    print(f"Loading file: {file}")  # Debugging: Print file being loaded
    df = pd.read_csv(file, delimiter=',', names=column_names, header=None, dtype=str)
    # print(df.head())  # Debugging: Print the first few rows of the loaded DataFrame
    data_list.append(df)

combined_data = pd.concat(data_list)

# Check the combined data before parsing dates
print("Combined data before parsing dates:")
print(combined_data.head())

# Combine <DATE> and <TIME> into a single datetime column
combined_data['datetime'] = pd.to_datetime(combined_data['<DATE>'] + ' ' + combined_data['<TIME>'], format='%Y.%m.%d %H:%M', errors='coerce')

# Check the combined data after parsing dates
print("Combined data after parsing dates:")
print(combined_data.head())

# Check for rows with NaT in datetime column
print("Rows with NaT in datetime column:")
print(combined_data[combined_data['datetime'].isna()].head())

# Drop rows with NaT in datetime column
combined_data.dropna(subset=['datetime'], inplace=True)

# Set datetime as the index
combined_data.set_index('datetime', inplace=True)
combined_data.sort_index(inplace=True)

# Check the combined data before converting numeric columns
print("Combined data before converting numeric columns:")
print(combined_data.head())

# Convert numeric columns to appropriate data types
numeric_columns = ['<OPEN>', '<HIGH>', '<LOW>', '<CLOSE>', '<TICKVOL>', '<VOL>', '<SPREAD>']
combined_data[numeric_columns] = combined_data[numeric_columns].apply(pd.to_numeric, errors='coerce')

# Check the combined data after converting numeric columns
print("Combined data after converting numeric columns:")
print(combined_data.head())

# Drop rows with NaN values in numeric columns
combined_data.fillna(0, inplace=True)

# Display the final combined data
print("Final combined data:")
combined_data.head()

Loading file: /root/github/CTI_Scripts/py_scripts/backtesting/historical_data/EURUSD/DAT_MT_EURUSD_M1_2021.csv
Loading file: /root/github/CTI_Scripts/py_scripts/backtesting/historical_data/EURUSD/DAT_MT_EURUSD_M1_2018.csv
Loading file: /root/github/CTI_Scripts/py_scripts/backtesting/historical_data/EURUSD/DAT_MT_EURUSD_M1_2002.csv
Loading file: /root/github/CTI_Scripts/py_scripts/backtesting/historical_data/EURUSD/DAT_MT_EURUSD_M1_2000.csv
Loading file: /root/github/CTI_Scripts/py_scripts/backtesting/historical_data/EURUSD/DAT_MT_EURUSD_M1_2017.csv
Loading file: /root/github/CTI_Scripts/py_scripts/backtesting/historical_data/EURUSD/DAT_MT_EURUSD_M1_2001.csv
Loading file: /root/github/CTI_Scripts/py_scripts/backtesting/historical_data/EURUSD/DAT_MT_EURUSD_M1_2003.csv
Loading file: /root/github/CTI_Scripts/py_scripts/backtesting/historical_data/EURUSD/DAT_MT_EURUSD_M1_2013.csv
Loading file: /root/github/CTI_Scripts/py_scripts/backtesting/historical_data/EURUSD/DAT_MT_EURUSD_M1_2020.csv
L

,<DATE>,<TIME>,<OPEN>,<HIGH>,<LOW>,<CLOSE>,<TICKVOL>,<VOL>,<SPREAD>
datetime,,,,,,,,,
2000-05-30 17:27:00,2000.05.30,17:27,0.9302,0.9302,0.9302,0.9302,0,0.0,0.0
2000-05-30 17:35:00,2000.05.30,17:35,0.9304,0.9305,0.9304,0.9305,0,0.0,0.0
2000-05-30 17:38:00,2000.05.30,17:38,0.9304,0.9304,0.9303,0.9303,0,0.0,0.0
2000-05-30 17:43:00,2000.05.30,17:43,0.9301,0.9301,0.9300,0.9300,0,0.0,0.0
2000-05-30 17:44:00,2000.05.30,17:44,0.9298,0.9298,0.9297,0.9297,0,0.0,0.0


In [5]:
import pandas_ta as ta

# Calculate indicators
combined_data['SMA_50'] = ta.sma(combined_data['<CLOSE>'], length=50)
combined_data['RSI'] = ta.rsi(combined_data['<CLOSE>'], length=14)

# Drop rows with NaN values (due to indicator calculation)
combined_data.dropna(inplace=True)

# Display the data with indicators
print("Data with indicators:")
combined_data.tail()

Data with indicators:


,<DATE>,<TIME>,<OPEN>,<HIGH>,<LOW>,<CLOSE>,<TICKVOL>,<VOL>,<SPREAD>,SMA_50,RSI
datetime,,,,,,,,,,,
2023-01-31 23:53:00,2023.01.31,23:53,1.08656,1.08659,1.08655,1.08659,0,0.0,0.0,1.086610,49.725059
2023-01-31 23:54:00,2023.01.31,23:54,1.08660,1.08660,1.08655,1.08656,0,0.0,0.0,1.086612,46.887511
2023-01-31 23:55:00,2023.01.31,23:55,1.08656,1.08658,1.08655,1.08656,0,0.0,0.0,1.086613,46.887511
2023-01-31 23:56:00,2023.01.31,23:56,1.08656,1.08667,1.08656,1.08663,0,0.0,0.0,1.086615,53.992206
2023-01-31 23:57:00,2023.01.31,23:57,1.08663,1.08672,1.08663,1.08672,0,0.0,0.0,1.086618,61.181925


In [ ]:
import backtrader as bt

# Define a custom strategy
class PandasTAStrategy(bt.Strategy):
    def __init__(self):
        self.sma_50 = bt.indicators.SimpleMovingAverage(self.data.close, period=50)
        self.rsi = bt.indicators.RelativeStrengthIndex(self.data.close, period=14)

    def next(self):
        if self.data.close[0] > self.sma_50[0] and self.rsi[0] < 30 and not self.position:
            self.buy()
        elif self.data.close[0] < self.sma_50[0] and self.rsi[0] > 70 and self.position:
            self.sell()

# Convert the pandas DataFrame to a Backtrader data feed
class PandasData(bt.feeds.PandasData):
    lines = ('sma_50', 'rsi',)
    params = (('sma_50', None), ('rsi', None),)

data_feed = PandasData(dataname=combined_data)

# Set up the Backtrader environment
cerebro = bt.Cerebro()
cerebro.addstrategy(PandasTAStrategy)
cerebro.adddata(data_feed)
cerebro.broker.set_cash(2500)
cerebro.broker.setcommission(commission=0.02)

# Run the backtest
cerebro.run()

# Plot the results
cerebro.plot()